In [ ]:
#==========================================================
# BLOQUE 1
# MONTAJE DE GOOGLE DRIVE Y LIBRERÍAS
#==========================================================

from google.colab import drive
drive.mount('/content/drive')
#==========================================================
# LIBRERÍAS
#==========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import warnings
warnings.filterwarnings("ignore")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#==========================================================
# BLOQUE 2
# CARGAR ARCHIVOS
#==========================================================

base_path = "/content/drive/MyDrive/Simulaciones_Mundial/Data"

datos_mundial = pd.read_csv(f"{base_path}/datos_mundial.csv")

datos_historicos = pd.read_csv(f"{base_path}/datos_historicos.csv")

partidos = pd.read_csv(f"{base_path}/partidos.csv")

grupos = pd.read_csv(f"{base_path}/Grupos_Mundial.csv", sep=';')

ranking = pd.read_csv(f"{base_path}/ranking_fifa.csv")

transfermarkt = pd.read_csv(f"{base_path}/transfermarkt.csv")
print("datos_mundial:", datos_mundial.shape)
print("datos_historicos:", datos_historicos.shape)
print("partidos:", partidos.shape)
print("grupos:", grupos.shape)
print("ranking:", ranking.shape)
print("transfermarkt:", transfermarkt.shape)

print(datos_mundial.columns.tolist()[:20])

print()

print(datos_historicos.columns.tolist()[:20])

datos_mundial: (48, 47)
datos_historicos: (1396, 45)
partidos: (3316, 2778)
grupos: (48, 2)
ranking: (6931, 3)
transfermarkt: (217, 2)
['Equipo', 'Puntos', 'Peso', 'Valor_Mercado_Millones_Eur', 'Tipo_Equipo', 'Fecha', 'Resultado_1X2', 'avg_Goles_esperados_(xG)_total', 'avg_Posesión_total', 'avg_Remates_a_puerta_total', 'avg_Córneres_total', 'avg_Tarjetas_amarillas_total', 'avg_Faltas_total', 'avg_Paradas_total', 'avg_Pases_Pct_total', 'avg_Pases_Exitosos_total', 'avg_Goles_total', 'avg_Ptos_total', 'avg_xG_Share_total', 'avg_Remates_Puerta_Share_total']

['Fecha', 'Equipo_Local', 'Equipo_Visitante', 'Goles_Local', 'Goles_Visitante', 'Peso_Local', 'avg_Goles_esperados_(xG)_total_Local', 'avg_Tarjetas_amarillas_total_Local', 'avg_Faltas_total_Local', 'avg_Remates_a_puerta_3_Local', 'avg_Córneres_3_Local', 'avg_Tarjetas_amarillas_3_Local', 'avg_Faltas_3_Local', 'avg_Paradas_3_Local', 'avg_Pases_Pct_3_Local', 'avg_Pases_Exitosos_3_Local', 'avg_Paradas_Share_3_Local', 'Peso_Visitante', 'Res

In [ ]:
# Elimina espacios en los nombres de las columnas
datos_mundial.columns = datos_mundial.columns.str.strip()
grupos.columns = grupos.columns.str.strip()
datos_historicos.columns = datos_historicos.columns.str.strip()
partidos.columns = partidos.columns.str.strip()

In [ ]:
#==========================================================
# BLOQUE 3
# NORMALIZACIÓN DE NOMBRES DE EQUIPOS
#==========================================================

# Diccionario de equivalencias
normalizar_equipo = {

    # América
    "USA": "Estados Unidos",
    "United States": "Estados Unidos",

    # Europa
    "England": "Inglaterra",
    "Netherlands": "Países Bajos",
    "Holland": "Países Bajos",
    "Switzerland": "Suiza",
    "Belgium": "Bélgica",
    "Germany": "Alemania",
    "Spain": "España",
    "Portugal": "Portugal",
    "France": "Francia",
    "Austria": "Austria",
    "Sweden": "Suecia",
    "Norway": "Noruega",
    "Bosnia & Herzegovina": "Bosnia y Herzegovina",
    "Bosnia and Herzegovina": "Bosnia y Herzegovina",

    # África
    "Ivory Coast": "Costa de Marfil",
    "South Africa": "Sudáfrica",
    "Cape Verde": "Cabo Verde",
    "Egypt": "Egipto",
    "Ghana": "Ghana",
    "Morocco": "Marruecos",

    # Asia
    "South Korea": "Corea del Sur",
    "Korea Republic": "Corea del Sur",
    "Japan": "Japón",
    "Iran": "Irán",

    # Oceanía
    "Australia": "Australia",

    # Sudamérica
    "Brazil": "Brasil",
    "Argentina": "Argentina",
    "Colombia": "Colombia",
    "Paraguay": "Paraguay",
    "Ecuador": "Ecuador",

    # Norteamérica
    "Canada": "Canadá",
    "Mexico": "México",
    "Senegal": "Senegal"
}

def normalizar_nombre(nombre):
    """
    Devuelve el nombre normalizado de un equipo.
    Si no existe en el diccionario, devuelve el original.
    """
    if pd.isna(nombre):
        return nombre

    nombre = str(nombre).strip()

    return normalizar_equipo.get(nombre, nombre)

    # datos_mundial
datos_mundial["Equipo"] = datos_mundial["Equipo"].apply(normalizar_nombre)

# históricos
datos_historicos["Equipo_Local"] = datos_historicos["Equipo_Local"].apply(normalizar_nombre)

datos_historicos["Equipo_Visitante"] = datos_historicos["Equipo_Visitante"].apply(normalizar_nombre)

# grupos
grupos['Equipo'] = grupos['Equipo'].apply(normalizar_nombre)

In [ ]:
print('Las columnas actuales del DataFrame `grupos` son:')
print(grupos.columns.tolist())

Las columnas actuales del DataFrame `grupos` son:
['Equipo', 'Grupo']


In [ ]:
#==========================================================
# BLOQUE 4
# CONSTRUCCIÓN AUTOMÁTICA DE FEATURES
#==========================================================

# Variables Local
features_local = [
    c for c in datos_historicos.columns
    if c.startswith("avg_") and c.endswith("_Local")
]

# Variables Visitante
features_visit = [
    c for c in datos_historicos.columns
    if c.startswith("avg_") and c.endswith("_Visitante")
]

print("Variables Local:", len(features_local))
print("Variables Visitante:", len(features_visit))

# Unimos todas las disponibles
X = datos_historicos[features_local + features_visit].copy()

# Targets
y_local = datos_historicos["Goles_Local"]
y_visitante = datos_historicos["Goles_Visitante"]

print("\nDimensiones de X:", X.shape)

Variables Local: 11
Variables Visitante: 13

Dimensiones de X: (1396, 24)


In [ ]:
#==========================================================
# LIMPIEZA
#==========================================================

# Eliminar partidos sin goles registrados
datos_validos = datos_historicos.dropna(
    subset=["Goles_Local", "Goles_Visitante"]
).copy()

# Reconstruir X e y con los datos válidos
X = datos_validos[features_local + features_visit]

y_local = datos_validos["Goles_Local"]
y_visitante = datos_validos["Goles_Visitante"]

# Rellenar valores faltantes con la media
X = X.fillna(X.mean(numeric_only=True))

print("X:", X.shape)
print("Nulos:", X.isnull().sum().sum())

X: (1394, 24)
Nulos: 0


In [ ]:
#==========================================================
# BLOQUE 5
# DIVISIÓN DEL CONJUNTO DE DATOS Y ESCALADO
#==========================================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# División entrenamiento/prueba
X_train, X_test, y_train_local, y_test_local = train_test_split(
    X,
    y_local,
    test_size=0.20,
    random_state=42
)

_, _, y_train_visit, y_test_visit = train_test_split(
    X,
    y_visitante,
    test_size=0.20,
    random_state=42
)

# Inicializar y ajustar el StandardScaler
scaler = StandardScaler()
scaler.fit(X_train)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)
print("✅ StandardScaler inicializado y ajustado.")

Entrenamiento: (1115, 24)
Prueba: (279, 24)
✅ StandardScaler inicializado y ajustado.


In [ ]:
#==========================================================
# RANDOM FOREST
#==========================================================

from sklearn.ensemble import RandomForestRegressor

rf_local = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_visit = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_local.fit(X_train, y_train_local)
rf_visit.fit(X_train, y_train_visit)

print("✅ Modelos entrenados correctamente.")

✅ Modelos entrenados correctamente.


In [ ]:
pred_local = rf_local.predict(X_test)
pred_visit = rf_visit.predict(X_test)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("========== LOCAL ==========")

print("MAE :", mean_absolute_error(y_test_local, pred_local))
print("RMSE:", np.sqrt(mean_squared_error(y_test_local, pred_local)))
print("R²  :", r2_score(y_test_local, pred_local))

print()

print("======= VISITANTE ========")

print("MAE :", mean_absolute_error(y_test_visit, pred_visit))
print("RMSE:", np.sqrt(mean_squared_error(y_test_visit, pred_visit)))
print("R²  :", r2_score(y_test_visit, pred_visit))

========== LOCAL ==========
MAE : 1.0121492085937913
RMSE: 1.3966400233903893
R²  : 0.16735078061591224

======= VISITANTE ========
MAE : 0.942234163373392
RMSE: 1.260837810423816
R²  : 0.2988885394640718


In [ ]:
#==========================================================
# BLOQUE 6
# FUNCIÓN PRINCIPAL DE PREDICCIÓN
#==========================================================
import pandas as pd
import random

def predecir_partido(equipo_local, equipo_visitante, fase_eliminatoria=False):
    """
    Predice el resultado entre dos equipos.
    Si fase_eliminatoria=True, resuelve los empates simulando penales o tiempo extra.
    """
    # 1. Normalizar nombres
    local_norm = normalizar_nombre(equipo_local)
    visit_norm = normalizar_nombre(equipo_visitante)

    # 2. Buscar estadísticas en datos_mundial
    try:
        data_local = datos_mundial[datos_mundial['Equipo'] == local_norm].iloc[0]
        data_visit = datos_mundial[datos_mundial['Equipo'] == visit_norm].iloc[0]
    except IndexError:
        raise ValueError(f"Error: Al menos uno de los equipos ('{local_norm}' o '{visit_norm}') no existe en datos_mundial.")

    # 3. Construir vector de entrada
    X_pred_dict = {}

    # Extraer variables del local
    for col in features_local:
        base_col = col.replace('_Local', '')
        X_pred_dict[col] = data_local.get(base_col, 0)

    # Extraer variables del visitante
    for col in features_visit:
        base_col = col.replace('_Visitante', '')
        X_pred_dict[col] = data_visit.get(base_col, 0)

    # Crear DataFrame asegurando el orden exacto de las columnas
    X_pred = pd.DataFrame([X_pred_dict])[X_train.columns]

    # Limpiar posibles valores faltantes
    X_pred = X_pred.fillna(X_pred.mean(numeric_only=True).fillna(0))

    # 4. Predecir goles (sin scaler, ya que rf se entrenó con X_train puro)
    goles_local_pred = rf_local.predict(X_pred)[0]
    goles_visit_pred = rf_visit.predict(X_pred)[0]

    # Redondeo para obtener goles enteros en el marcador final
    goles_local = int(round(goles_local_pred))
    goles_visit = int(round(goles_visit_pred))

    # 5. Determinar ganador y resolver empates
    if goles_local > goles_visit:
        ganador = local_norm
    elif goles_visit > goles_local:
        ganador = visit_norm
    else:
        if fase_eliminatoria:
            # Regla de desempate usando los decimales de la predicción como ventaja
            if goles_local_pred > goles_visit_pred:
                ganador = local_norm
            elif goles_visit_pred > goles_local_pred:
                ganador = visit_norm
            else:
                # Desempate total al azar (simulando tanda de penales impredecible)
                ganador = random.choice([local_norm, visit_norm])
        else:
            ganador = "Empate"

    # Mostrar resultados en consola
    print(f"Predicción: {local_norm} vs {visit_norm}")
    print(f"Marcador: {goles_local} - {goles_visit}")
    print(f"Ganador: {ganador}")
    if fase_eliminatoria and goles_local == goles_visit:
        print("(Victoria decidida en desempate/penales)")
    print("-" * 30)

    return {
        "Local": local_norm,
        "Visitante": visit_norm,
        "Goles_Local": goles_local,
        "Goles_Visitante": goles_visit,
        "Ganador": ganador
    }

# Prueba de la función
resultado = predecir_partido("Brasil", "España", fase_eliminatoria=True)

Predicción: Brasil vs España
Marcador: 1 - 2
Ganador: España
------------------------------
